In [1]:
# import libraries for reading data
import pandas as pd
import matplotlib.pyplot as plt
import os
from PIL import Image
import cv2
import re
import torch
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from torchvision import transforms, models
import ast
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
import torch.nn as nn
from sklearn.model_selection import train_test_split
from concurrent.futures import ProcessPoolExecutor
from PIL import ImageEnhance


<jemalloc>: Unsupported system page size


### Einlesen der Daten und Übersicht über die Daten

In [2]:
# data paths
train1_images_path = "/datasets/multi-view-pig-posture-recognition/train1_images"
train2_images_path = "/datasets/multi-view-pig-posture-recognition/train2_images"
test_images_path = "/datasets/multi-view-pig-posture-recognition/test_images"

# csv path with row_id, image_id, width, height, bbox, class_id
train1_csv_path = "/datasets/multi-view-pig-posture-recognition/train1.csv"
train2_csv_path = "/datasets/multi-view-pig-posture-recognition/train2.csv"
test_csv_path = "/datasets/multi-view-pig-posture-recognition/test.csv"

# txt file path
pig_posture_txt = "/datasets/multi-view-pig-posture-recognition/pig_posture_classes.txt"

In [3]:
# read all files and show statistics and content of csv files, column names etc.
# read csv files
train1_df = pd.read_csv(train1_csv_path)
train2_df = pd.read_csv(train2_csv_path)
test_df = pd.read_csv(test_csv_path)

# show column names of csv files
print("\nTrain1 CSV Columns:")
print(train1_df.columns)
print("\nTrain2 CSV Columns:")
print(train2_df.columns)
print("\nTest CSV Columns:")
print(test_df.columns)

# show content of txt file
with open(pig_posture_txt, 'r') as f:
    pig_posture_content = f.read()

# show numbers of unique image_ids, row_ids in train1, train2 and test csv files
print("\nNumber of unique image_ids in Train1 CSV:", train1_df['image_id'].nunique())
print("Number of unique image_ids in Train2 CSV:", train2_df['image_id'].nunique())
print("Number of unique row_ids in Train1 CSV:", train1_df['row_id'].nunique())
print("Number of unique row_ids in Train2 CSV:", train2_df['row_id'].nunique())
print("\nPig Posture Classes:")
print(pig_posture_content)




Train1 CSV Columns:
Index(['row_id', 'image_id', 'width', 'height', 'bbox', 'class_id'], dtype='object')

Train2 CSV Columns:
Index(['row_id', 'image_id', 'width', 'height', 'bbox', 'class_id'], dtype='object')

Test CSV Columns:
Index(['row_id', 'image_id', 'width', 'height', 'bbox'], dtype='object')

Number of unique image_ids in Train1 CSV: 3090
Number of unique image_ids in Train2 CSV: 3150
Number of unique row_ids in Train1 CSV: 22934
Number of unique row_ids in Train2 CSV: 23450

Pig Posture Classes:
Lateral_lying_left
Lateral_lying_right
Sitting
Standing
Sternal_lying



In [4]:
train1_df.head()

,row_id,image_id,width,height,bbox,class_id
0,train_pen1_orb_cam1_20250108_085204_0000,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1031.5,368.0,349.0,435.0]",0
1,train_pen1_orb_cam1_20250108_085204_0001,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1278.5,428.0,233.0,438.0]",4
2,train_pen1_orb_cam1_20250108_085204_0002,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[732.0,137.5,342.0,198.0]",1
3,train_pen1_orb_cam1_20250108_085204_0003,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[830.0,169.0,370.0,263.0]",0
4,train_pen1_orb_cam1_20250108_085204_0004,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[611.5,314.8,381.5,386.6]",3


In [5]:
train2_df.head()

,row_id,image_id,width,height,bbox,class_id
0,train_pen1_orb_cam1_20250108_085204_0000,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1031.5,368.0,349.0,435.0]",0
1,train_pen1_orb_cam1_20250108_085204_0001,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1278.5,428.0,233.0,438.0]",4
2,train_pen1_orb_cam1_20250108_085204_0002,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[732.0,137.5,342.0,198.0]",1
3,train_pen1_orb_cam1_20250108_085204_0003,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[830.0,169.0,370.0,263.0]",0
4,train_pen1_orb_cam1_20250108_085204_0004,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[611.5,314.8,381.5,386.6]",3


### EDA 
Class Definitions:

0 — Lateral_lying_left

1 — Lateral_lying_right

2 — Sitting

3 — Standing

4 — Sternal_lying

### Klassen sind stark unausgewogen, insbesondere Sitting Class id = 2. Gegenmaßnahme ist notwendig, um die Minderheitsklasse nicht zu vernachlässigen.

In [6]:
# check if there are any missing values in train1 and train2 csv files
print("\nMissing values in Train1 CSV:")
print(train1_df.isnull().sum())
print("\nMissing values in Train2 CSV:")
print(train2_df.isnull().sum())


Missing values in Train1 CSV:
row_id      0
image_id    0
width       0
height      0
bbox        0
class_id    0
dtype: int64

Missing values in Train2 CSV:
row_id      0
image_id    0
width       0
height      0
bbox        0
class_id    0
dtype: int64


In [7]:
# check if unique values in "height" and "weight" columns in train1 and train2 csv files are the same
print("\nUnique values in 'height' column in Train1 CSV:")
print(train1_df['height'].unique())
print("\nUnique values in 'height' column in Train2 CSV:")
print(train2_df['height'].unique())
print("\nUnique values in 'width' column in Train1 CSV:")
print(train1_df['width'].unique())
print("\nUnique values in 'width' column in Train2 CSV:")
print(train2_df['width'].unique())


Unique values in 'height' column in Train1 CSV:
[1080  720 1520]

Unique values in 'height' column in Train2 CSV:
[1080  720 1520]

Unique values in 'width' column in Train1 CSV:
[1920 1280 2688]

Unique values in 'width' column in Train2 CSV:
[1920 1280 2688]


### Die Bilder liegen nur in drei Auflösungen vor: 1280 x 720, 1920 x 1080, 2688 x 1520. Vorverarbeitung ist konsistent planbar. 

In [8]:
# Blur-Score (OpenCV)
def blur_score(img_bgr):
    return cv2.Laplacian(img_bgr, cv2.CV_64F).var()

# Helligkeit in HSV
def mean_brightness(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    return float(hsv[...,2].mean())


### Hinzufügen von der Spalte brightness_score und blur_score für die Weiterverarbeitung

In [9]:
# Diese Funktion verarbeitet ein einzelnes Bild extrem schnell
# Diese Funktion verarbeitet ein einzelnes Bild extrem schnell
def get_fast_metrics(args):
    img_id, path = args
    full_path = os.path.join(path, img_id)
    
    # TRICK: IMREAD_REDUCED_COLOR_4 liest das Bild direkt in 1/4 der Größe ein.
    # Das ist ca. 16x schneller und reicht für Blur/Helligkeit völlig aus!
    img = cv2.imread(full_path, cv2.IMREAD_REDUCED_COLOR_4)
    
    if img is not None:
        # Hier deine vorhandenen Funktionen blur_score und mean_brightness nutzen
        return blur_score(img), mean_brightness(img)
    return 0, 0

def fast_add_metrics(df, images_path, workers=16):
    # Aufgaben vorbereiten
    tasks = [(row['image_id'], images_path) for _, row in df.iterrows()]
    
    # Parallelisierung über alle verfügbaren Kerne
    with ProcessPoolExecutor(max_workers=workers) as executor:
        # chunksize verhindert, dass die Kommunikation zwischen den Kernen zum Flaschenhals wird
        results = list(tqdm(executor.map(get_fast_metrics, tasks, chunksize=100), 
                           total=len(tasks), 
                           desc=f"Blitz-Metriken: {os.path.basename(images_path)}"))
    
    # Ergebnisse effizient in den DataFrame schreiben
    df['blur_score'], df['mean_brightness'] = zip(*results)
    return df

# Anwendung (num_workers=16 ist sicher, bei V100-Servern gehen oft auch 32)
train1_df = fast_add_metrics(train1_df, train1_images_path, workers=16)
train2_df = fast_add_metrics(train2_df, train2_images_path, workers=16)

Blitz-Metriken: train2_images: 100%|██████████| 23450/23450 [00:42<00:00, 553.29it/s] 


In [10]:
train1_df = train1_df[train1_df['blur_score'] >= 100].copy()
train2_df = train2_df[train2_df['blur_score'] >= 100].copy()

### Fazit: Die EDA zeigt, dass die Klassen in den Trainingsdaten relativ ausgewogen verteilt sind, was für das Training eines Modells vorteilhaft ist. Es gibt keine fehlenden Werte in den CSV-Dateien, und die Bildgrößen sind konsistent. Die Analyse der Bildqualität anhand von Blur-Score und Helligkeit zeigt eine gewisse Variation. Im nächsten Schritt möchte ich die Kameras trennen und die Bilder entsprechend der Kamera analysieren, um mögliche Unterschiede in der Bildqualität oder den Aufnahmewinkeln zu identifizieren.

### Trennung der Kameras anhand der Bildnamen nur in Train1

In [11]:
def extract_pen_id(s: str):
    m = re.search(r'^(pen\d+)', s)
    return m.group(1) if m else None

def extract_camera_type(s: str):
    m = re.search(r'_(orb|tur)_', s)
    return m.group(1) if m else None

def extract_camera_number(s: str):
    m = re.search(r'cam(\d+)', s)
    return m.group(1) if m else None

# df1 = train1.csv als DataFrame; ersetze 'FILENAME_COL' durch deine Spalte (z. B. 'image', 'file_name', ...).
FILENAME_COL = "image_id"
train1_df["pen_id"]        = train1_df[FILENAME_COL].apply(extract_pen_id)
train1_df["camera_type"]   = train1_df[FILENAME_COL].apply(extract_camera_type)
train1_df["camera_number"] = train1_df[FILENAME_COL].apply(extract_camera_number)
train1_df["camera_view_id"]      = train1_df["pen_id"] + "_" + train1_df["camera_type"] + "_cam" + train1_df["camera_number"]


In [12]:
train1_df_distribution = train1_df['class_id'].value_counts().sort_index()
train2_df_distribution = train2_df['class_id'].value_counts().sort_index()

print("\nClass Distribution in Train1 CSV:")
print(train1_df_distribution)
print("\nClass Distribution in Train2 CSV:")
print(train2_df_distribution)


Class Distribution in Train1 CSV:
class_id
0    3053
1    3376
2     680
3    9617
4    6208
Name: count, dtype: int64

Class Distribution in Train2 CSV:
class_id
0    3083
1    3435
2     695
3    9928
4    6309
Name: count, dtype: int64


### Trennung der Kameras anhand der Bildnamen nur in Train2

In [13]:
train2_df["pen_id"]        = train2_df["image_id"].apply(extract_pen_id)
train2_df["camera_type"]   = train2_df["image_id"].apply(extract_camera_type)
train2_df["camera_number"] = train2_df["image_id"].apply(extract_camera_number)
train2_df["camera_view_id"]      = train2_df["pen_id"] + "_" + train2_df["camera_type"] + "_cam" + train2_df["camera_number"]


### Erster Versuch eines Modelltrainings mit dem Modell "MobileNetv3". Vorbereitungen treffen mit transforms. 

In [14]:
# transforms for data augmentation and data preprocessing
img_size = (224, 224)
normalize_mean = [0.485, 0.456, 0.406]
normalize_std = [0.229, 0.224, 0.225]


train_transforms = transforms.Compose([
    transforms.Resize(img_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std),
])

val_transforms = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std),
])



In [15]:

class PigCropDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.cache = [None] * len(self.df)  # Speicherplatz im RAM reservieren
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Wenn im RAM vorhanden, nimm das (Bausatz-Prinzip)
        if self.cache[idx] is not None:
            crop_img, label = self.cache[idx]
        else:
            # Ansonsten: Einmalig den schweren Prozess durchlaufen (Laden & Schneiden)
            r = self.df.iloc[idx]
            p = os.path.join(self.image_dir, r["image_id"])
            image = Image.open(p).convert("RGB")
            
            bbox = ast.literal_eval(r["bbox"]) if isinstance(r["bbox"], str) else r["bbox"]
            x1, y1, w, h = bbox
            # PIL .crop ist schneller als numpy-slicing für diesen Zweck
            crop_img = image.crop((x1, y1, x1 + w, y1 + h))

            brightness = r.get('mean_brightness', 100)
            if r['mean_brightness'] < 80:
                enhancer = ImageEnhance.Brightness(crop_img)
                # Faktor 1.5 macht es 50% heller
                crop_img = enhancer.enhance(1.5)
            
            label = int(r["class_id"])
            
            # Im RAM speichern für die nächste Epoche
            self.cache[idx] = (crop_img, label)

        # Transformationen (Augmentation) immer erst NACH dem Cache anwenden,
        # damit jede Epoche ein leicht anderes Bild sieht (Training wird besser).
        if self.transform:
            return self.transform(crop_img), torch.tensor(label, dtype=torch.long)
        
        return crop_img, torch.tensor(label, dtype=torch.long)



In [16]:
sampleDS = PigCropDataset(train2_df, train2_images_path, transform=train_transforms)
loader = DataLoader(sampleDS, batch_size=9, shuffle=True)
for x, y in loader:
    print("Batch Image Shape:", x.shape)  # erwartet: [9, 3, 224, 224]
    print("Batch Label Shape:", y.shape)  # erwartet: [9]
    print("First Label:", y[0].item())    # 0..4
    break


Batch Image Shape: torch.Size([9, 3, 224, 224])
Batch Label Shape: torch.Size([9])
First Label: 1


In [17]:
class PigPostureCNN(nn.Module):
    def __init__(self, num_classes=5):
        super(PigPostureCNN, self).__init__()
        
        # Backbone: MobileNetV3 statt ResNet18
        weights = MobileNet_V3_Large_Weights.DEFAULT
        original_model = mobilenet_v3_large(weights=weights)
        
        # Features und Pooling extrahieren
        self.backbone = nn.Sequential(
            original_model.features,
            original_model.avgpool
        )
        
        # Backbone einfrieren (wie bei der Lehrkraft)
        for param in self.backbone.parameters():
            param.requires_grad = False
        
        # Label_Classifier (Struktur der Lehrkraft beibehalten)
        in_features = original_model.classifier[0].in_features
        self.label_classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        label_preds = self.label_classifier(x)
        return label_preds

# Instanziierung auf cuda:3
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
model = PigPostureCNN(num_classes=5).to(device)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /home/jovyan/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


  0%|          | 0.00/21.1M [00:00<?, ?B/s]

### Einfügen einer Metrik, um mit der Klassenverteilung besser ausheben zu können. Den unterrepräsentierten Klassen eine höhere Gewichtung geben, um die Ungleichheit der Klassenverteilung zu adressieren. 

In [18]:
# 2. Die neue Verteilung berechnen
new_counts = train2_df['class_id'].value_counts().sort_index()
print("Neue Verteilung nach Blur-Filter:")
print(new_counts)

# 3. Gewichte dynamisch berechnen
# Wir nehmen die Anzahl der Bilder pro Klasse als Array
counts_array = new_counts.values
max_val = counts_array.max()

# Formel: w_i = max_n / n_i
weights_dynamic = torch.tensor([max_val / c for c in counts_array], dtype=torch.float32).to(device)

print(f"\nDynamisch berechnete Gewichte für den Learner: {weights_dynamic}")

Neue Verteilung nach Blur-Filter:
class_id
0    3083
1    3435
2     695
3    9928
4    6309
Name: count, dtype: int64

Dynamisch berechnete Gewichte für den Learner: tensor([ 3.2202,  2.8902, 14.2849,  1.0000,  1.5736], device='cuda:3')


In [19]:
class Learner:
    def __init__(self, model, train_dl, val_dl, device=None):
        self.model = model
        self.train_dl = train_dl
        self.val_dl = val_dl
        self.device = device
        
        self.model = self.model.to(self.device)
        self.loss_fn_classifier = nn.CrossEntropyLoss(weight = weights_dynamic) # neue Verlustfunktion mit Klassen-Gewichtung aus der obigen Berechnung
        self.best_acc = 0
        self.freeze()
        
    def freeze(self):
        for param in self.model.backbone.parameters():
            param.requires_grad = False
        
    def unfreeze(self):
        for param in self.model.backbone.parameters():
            param.requires_grad = True

    def fit(self, epochs, lr=1e-3):
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr)
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimizer, max_lr=lr*10, total_steps=epochs*len(self.train_dl)
        )
        
        for epoch in range(epochs):
            self.model.train()
            for xb, yb in tqdm(self.train_dl, desc=f"Epoch {epoch+1}"):
                xb, yb = xb.to(self.device), yb.to(self.device)
                
                self.optimizer.zero_grad()
                preds = self.model(xb)
                loss = self.loss_fn_classifier(preds, yb)
                loss.backward()
                self.optimizer.step()
                self.scheduler.step()
            
            # Validierung
            self.model.eval()
            correct = 0
            with torch.no_grad():
                for xb, yb in self.val_dl:
                    xb, yb = xb.to(self.device), yb.to(self.device)
                    preds = self.model(xb)
                    correct += (preds.argmax(1) == yb).sum().item()
            
            acc = correct / len(self.val_dl.dataset)
            if acc > self.best_acc: self.best_acc = acc
            print(f"Validation Accuracy: {acc:.4f}")

In [20]:
train_data, val_data = train_test_split(
    train2_df, 
    test_size=0.15, 
    random_state=42, 
    stratify=train2_df['class_id']
)

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

Training samples: 19932
Validation samples: 3518


In [21]:
# --- VORBEREITUNG ---
# Nutze hier die Version von PigCropDataset, in die wir das Caching (RAM-Speicher) 
# eingebaut haben, damit es ab Epoche 2 extrem schnell geht.
train_ds = PigCropDataset(train_data, train2_images_path, transform=train_transforms)
val_ds = PigCropDataset(val_data, train2_images_path, transform=val_transforms)

# DataLoaders - Optimierung: pin_memory auch für Validierung nutzen
batch_size = 128
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=16, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)

# --- TRAINING STARTEN ---
# 1. Modell instanziieren
model = PigPostureCNN(num_classes=5).to(device)

# 2. Learner initialisieren
learner = Learner(model, train_dl, val_dl, device=device)

# 3. Sicherstellen, dass der Backbone eingefroren ist (WICHTIG für Phase 1)
learner.freeze()

# 4. Erste 10 Epochen trainieren (Classifier lernt die Grundlagen)
learner.fit(epochs=10, lr=1e-3)

Epoch 1: 100%|██████████| 156/156 [00:50<00:00,  3.11it/s]


Validation Accuracy: 0.6987


Epoch 2: 100%|██████████| 156/156 [00:55<00:00,  2.80it/s]


Validation Accuracy: 0.7078


Epoch 3: 100%|██████████| 156/156 [00:56<00:00,  2.75it/s]


...

Validation Accuracy: 0.7717


Epoch 9: 100%|██████████| 156/156 [00:54<00:00,  2.85it/s]


Validation Accuracy: 0.7752


Epoch 10: 100%|██████████| 156/156 [00:55<00:00,  2.79it/s]


Validation Accuracy: 0.7774


In [22]:
# --- PHASE 2: FINE-TUNING ---
# Jetzt tauen wir das Modell auf, um die Details für den Kaggle-Score zu lernen
learner.unfreeze()

# Wir trainieren weiter, aber mit einer VIEL kleineren Lernrate
learner.fit(epochs=20, lr=1e-5)

Epoch 1: 100%|██████████| 156/156 [00:59<00:00,  2.61it/s]


Validation Accuracy: 0.7862


Epoch 2: 100%|██████████| 156/156 [01:02<00:00,  2.49it/s]


Validation Accuracy: 0.7945


Epoch 3: 100%|██████████| 156/156 [01:03<00:00,  2.45it/s]


...

Validation Accuracy: 0.9341


Epoch 19: 100%|██████████| 156/156 [01:17<00:00,  2.02it/s]


Validation Accuracy: 0.9338


Epoch 20: 100%|██████████| 156/156 [01:22<00:00,  1.89it/s]


Validation Accuracy: 0.9341


In [23]:
# 1. Test-Daten vorbereiten
# Wir fügen eine Dummy-Spalte hinzu, damit dein PigCropDataset nicht abstürzt
test_df_copy = test_df.copy()
test_df_copy['class_id'] = 0 

# Dataset und Loader für Testdaten (Identisch zu deinem Training)
test_ds = PigCropDataset(test_df_copy, test_images_path, transform=val_transforms)
test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True)

# 2. Modell in den Vorhersage-Modus schalten
model.eval()
all_predictions = []

print("Erstelle Vorhersagen auf cuda:3...")
with torch.no_grad():
    for xb, _ in tqdm(test_dl):
        xb = xb.to(device) # Schiebt Daten auf cuda:3
        
        # Vorhersage berechnen
        outputs = model(xb)
        
        # Die Klasse mit dem höchsten Wert auswählen (0 bis 4)
        _, preds = torch.max(outputs, 1)
        
        # Ergebnisse sammeln
        all_predictions.extend(preds.cpu().numpy())

# 3. Die finale CSV-Datei erstellen
submission = pd.DataFrame({
    'row_id': test_df['row_id'],
    'class_id': all_predictions
})

# Als CSV speichern (ohne Index, wie von Kaggle verlangt)
submission.to_csv('2nd_submission.csv', index=False)

print(f"Erfolgreich! Die Datei 'submission.csv' mit {len(submission)} Zeilen wurde erstellt.")

Erstelle Vorhersagen auf cuda:3...


  0%|          | 0/92 [00:00<?, ?it/s]


KeyError: Caught KeyError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/opt/conda/envs/torch/lib/python3.10/site-packages/pandas/core/indexes/base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'mean_brightness'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/conda/envs/torch/lib/python3.10/site-packages/torch/utils/data/_utils/worker.py", line 302, in _worker_loop
    data = fetcher.fetch(index)
  File "/opt/conda/envs/torch/lib/python3.10/site-packages/torch/utils/data/_utils/f

### Erste Submission bei 10 Epochen hat eine Accuracy von 0.310 bei Kaggle erreicht. Erheblich von dem entfernt, was hier als Validation Accuracy von 0.83 zuletzt angezeigt wird

### Optimisierungsmaßnahmen, um einen besseren Score zu erreichen. Problematik der Verteilung der Klassen, unscharfe Bilder und Helligkeit erhöhen.